# Debt browser
Every instrument in the latest `cache/debt/debt_*.parquet` snapshot (`pricing/shared/debt_snapshot.py run`), filterable by CDS ticker, name text, currency, asset class, rank and size.
Needs the **system Python** kernel (ipywidgets / itables are not in the project .venv). Reads the file only; no Bloomberg.

In [ ]:
import pathlib
import pandas as pd
import ipywidgets as w
from IPython.display import display, clear_output, HTML

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'cache' / 'debt').exists())
MIN_5Y_BP = 200      # keep only names whose 5y CDS spread is at least this (bp); 0 keeps everything. Others are dropped here, so they never reach the tables
FILE = sorted((ROOT / 'cache' / 'debt').glob('debt_*.parquet'))[-1]          # newest snapshot
df = pd.read_parquet(FILE).sort_values(['market_cds_ticker', 'maturity'], na_position='last').reset_index(drop=True)
# 5Y CDS spread (bp, conventional at the index coupon): latest published date in data/base, the ~5y point (maturity 4.5-5.6y out; names not in a current index stay blank)
PR = ROOT / 'data' / 'base' / 'prices'
_s = pd.concat([pd.read_parquet(PR / f, columns=['market_cds_ticker', 'published_date', 'cds_maturity', 'conv_spread']) for f in ('priced_at_index_coupon.parquet', 'addin_fill.parquet')])
_s = _s[_s.published_date == _s.published_date.max()]
_s = _s[((_s.cds_maturity - _s.published_date).dt.days / 365.25).between(4.5, 5.6) & (_s.conv_spread > 0)]
SPREAD_DATE = _s.published_date.max()
df['5y_cds_spread'] = df.market_cds_ticker.map(_s.sort_values('cds_maturity').drop_duplicates('market_cds_ticker', keep='last').set_index('market_cds_ticker').conv_spread).round(1)
_fx = (df.notional_outstanding / df.notional_local).where(df.notional_local > 0).groupby(df.currency).median()      # USD per unit of each currency, from rows that have both amounts (so undrawn facilities can be converted too)
df['avail_usd_mm'] = (df.notional_total * df.currency.map(_fx) / 1e6)                                                 # 'available' in USD, for the summary line only
# once the snapshot has Bloomberg's security_type (next fetch), RCF comes from it; older snapshots fall back to the facility word in the name
_tok = df.security_name.str.split().str[1].fillna('')      # Bloomberg loan names are '<TICKER> <FACILITY> <RANK> <CCY> Corp': REV / RBL-REV / STANDBY-REV = revolver
df['kind'] = 'Bonds'
df.loc[df.asset_class.eq('Loans'), 'kind'] = 'Loans'
_rcf = df.security_type.isin(['REV']) if 'security_type' in df.columns else _tok.str.endswith('REV')
df.loc[df.asset_class.eq('Loans') & _rcf, 'kind'] = 'RCF'
if MIN_5Y_BP:
    df = df[df['5y_cds_spread'] >= MIN_5Y_BP].reset_index(drop=True)      # also drops names with no 5y spread
df['amt_out_usd'] = (df.notional_outstanding / 1e6).round(1)      # all amounts in mm
df['amt_out'] = (df.notional_local / 1e6).round(1)                # own currency
df['available'] = (df.notional_total / 1e6).round(1)                  # loans: tranche size; bonds: same as amt_out
df['amt_out_%'] = (df.notional_local / df.notional_total * 100).where(df.notional_total > 0).round(1)      # amt_out / available, in percent
NUM = {'5y_cds_spread': '{:,.1f}', 'last_price': '{:,.2f}', 'coupon': '{:,.3f}', 'amt_out_usd': '{:,.1f}', 'amt_out': '{:,.1f}', 'available': '{:,.1f}', 'amt_out_%': '{:,.1f}'}      # comma-separated number columns (amounts in mm)
SHOW = {'market_cds_ticker': 'cds_ticker'}      # display name only
COLS = ['market_cds_ticker', '5y_cds_spread', 'security_name', 'last_price', 'asset_class', 'payment_rank', 'currency', 'issue_date', 'maturity', 'coupon', 'amt_out_usd', 'amt_out', 'available', 'amt_out_%', 'amt_out_asof', 'security_id', 'equity_ticker']
print(f'5y_cds_spread as of {SPREAD_DATE:%Y-%m-%d}: {df["5y_cds_spread"].notna().sum():,} instruments / {df.loc[df["5y_cds_spread"].notna(), "market_cds_ticker"].nunique()} names have one')
print(f'kept 5y spread >= {MIN_5Y_BP} bp (MIN_5Y_BP)' if MIN_5Y_BP else 'no 5y spread cut')
print(f'{FILE.name}: {len(df):,} instruments, {df.market_cds_ticker.nunique()} names, pulled {df.pulled_at.max():%Y-%m-%d %H:%M}')

In [ ]:
MAX_ROWS = 100      # rows shown in the table below (the summary line still covers every matching instrument)
ticker = w.SelectMultiple(options=sorted(df.market_cds_ticker.unique()), rows=8, description='Ticker', layout=w.Layout(width='260px'))
ticker_txt = w.Text(description='Ticker starts', placeholder='e.g. MP, HL', layout=w.Layout(width='260px'))
name_txt = w.Text(description='Name has', placeholder='e.g. TL B, 5 10/15', layout=w.Layout(width='260px'))
ccy = w.SelectMultiple(options=sorted(df.currency.dropna().unique()), rows=5, description='Currency', layout=w.Layout(width='160px'))
cls = w.SelectMultiple(options=sorted(df.asset_class.dropna().unique()), rows=3, description='Class', layout=w.Layout(width='200px'))
rank = w.SelectMultiple(options=sorted(df.payment_rank.dropna().unique()), rows=5, description='Rank', layout=w.Layout(width='260px'))
min_usd = w.FloatText(value=0, description='Min $mm', layout=w.Layout(width='160px'))
kind = w.RadioButtons(options=['All', 'Bonds', 'Loans', 'RCF'], value='All', description='Type', layout=w.Layout(width='140px'))      # one choice only; Loans = loans other than revolvers
min_5y = w.FloatText(value=0, description='Min 5y bp', layout=w.Layout(width='160px'))
hide_zero = w.Checkbox(value=False, description='Hide undrawn / zero amount')
hide_matured = w.Checkbox(value=True, description='Hide matured (perpetuals kept)')
out = w.HTML()      # an HTML widget (value is replaced, never appended), so a refresh can't stack up copies of the table

def refresh(*_):
    d = df
    if ticker.value:
        d = d[d.market_cds_ticker.isin(ticker.value)]
    if ticker_txt.value:
        d = d[d.market_cds_ticker.str.upper().str.startswith(ticker_txt.value.strip().upper())]
    if name_txt.value:
        d = d[d.security_name.str.contains(name_txt.value.strip(), case=False, regex=False, na=False)]
    if ccy.value:
        d = d[d.currency.isin(ccy.value)]
    if cls.value:
        d = d[d.asset_class.isin(cls.value)]
    if rank.value:
        d = d[d.payment_rank.isin(rank.value)]
    if min_usd.value:
        d = d[d.amt_out_usd >= min_usd.value]
    if min_5y.value:
        d = d[d['5y_cds_spread'] >= min_5y.value]
    if kind.value != 'All':
        d = d[d.kind == kind.value]
    if hide_matured.value:
        d = d[d.maturity.isna() | (d.maturity >= pd.Timestamp.today().normalize())]
    if hide_zero.value:
        d = d[d.amt_out_usd > 0]
    summary = f'{len(d):,} instruments, {d.market_cds_ticker.nunique()} names, USD {d.amt_out_usd.sum():,.0f}mm outstanding, {d.avail_usd_mm.sum():,.0f}mm available'
    note = f'<p>(showing the first {MAX_ROWS} of {len(d):,}: narrow the filters)</p>' if len(d) > MAX_ROWS else ''
    fmt = {c: (lambda v, f=f: f.format(v)) for c, f in NUM.items()}
    table = d[COLS].head(MAX_ROWS).rename(columns=SHOW).to_html(index=False, classes='dbt', na_rep='', formatters=fmt)      # to_html: no jinja2 needed
    out.value = '<style>.dbt th, .dbt td {text-align: center !important; font-size: 90% !important;}.dbt td {padding-top: 1px !important; padding-bottom: 1px !important; line-height: 1.1 !important;}</style>' + f'<p>{summary}</p>' + table + note

for c in (ticker, ticker_txt, name_txt, ccy, cls, rank, min_usd, min_5y, kind, hide_zero, hide_matured):
    c.observe(refresh, names='value')
display(w.VBox([w.HBox([w.VBox([ticker_txt, ticker]), w.VBox([name_txt, min_usd, min_5y, hide_matured, hide_zero]), kind, ccy, cls, rank]), out]))
refresh()

Full table (sortable, searchable in the page) via itables, if you prefer a search box over the widgets:

In [ ]:
from itables import show
from itables.javascript import JavascriptFunction as JS
show(df[df.maturity.isna() | (df.maturity >= pd.Timestamp.today().normalize())][COLS].rename(columns=SHOW), maxBytes=0, showIndex=False, columnDefs=[{'className': 'dt-center', 'targets': '_all'}] + [{'targets': [COLS.index(c)], 'render': JS(f"function(d, t) {{ return t === 'display' && d !== null ? Number(d).toLocaleString('en-US', {{minimumFractionDigits: {NUM[c].split('.')[-1][0]}, maximumFractionDigits: {NUM[c].split('.')[-1][0]}}}) : d; }}")} for c in NUM], lengthMenu=[25, 100, 500], classes='display compact')